In [1]:
# Kaggle setup
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_32feae0c86a02acb063a9c9873d0afeb'

!pip install -q kaggle
!mkdir -p ~/.kaggle
!echo '{"KAGGLE_API_TOKEN":"'$KAGGLE_API_TOKEN'"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d adityajn105/flickr8k
!unzip -q flickr8k.zip -d flickr8k_data
print("Dataset download complete!")

Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
100% 1.04G/1.04G [00:09<00:00, 118MB/s] 

Dataset download complete!


In [2]:
# Install evaluation libraries
!pip install -q transformers accelerate pillow evaluate nltk rouge_score

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import pandas as pd
import random

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

print("BLIP model successfully loaded!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
Using device: cpu


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

BLIP model successfully loaded!


In [3]:
image_folder = "flickr8k_data/Images"
caption_file = "flickr8k_data/captions.txt"

captions_df = pd.read_csv(caption_file)

random.seed(42)
all_images = os.listdir(image_folder)
validation_images = random.sample(all_images, 200)   # 200 images ke liye; 500 karna ho to yahan number badal dein

print("Total validation images selected:", len(validation_images))
print("First 5:", validation_images[:5])

Total validation images selected: 200
First 5: ['2594336381_a93772823b.jpg', '3619806638_7480883039.jpg', '2550109269_bc4262bd27.jpg', '3195969533_98f5de0fab.jpg', '314940358_ec1958dc1d.jpg']


In [4]:
from tqdm import tqdm

def generate_caption(image_path):
    raw_image = Image.open(image_path).convert("RGB")
    inputs = processor(raw_image, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=30)
    caption = processor.decode(out[0], skip_special_tokens=True)
    return caption

blip_predictions = {}

for img_name in tqdm(validation_images, desc="Generating captions"):
    img_path = os.path.join(image_folder, img_name)
    caption = generate_caption(img_path)
    blip_predictions[img_name] = caption

print("\nDone! Generated captions for", len(blip_predictions), "images")
print("\nSample predictions:")
for img in validation_images[:3]:
    print(f"{img}: {blip_predictions[img]}")

Generating captions: 100%|██████████| 200/200 [11:48<00:00,  3.54s/it]


Done! Generated captions for 200 images

Sample predictions:
2594336381_a93772823b.jpg: a boy sitting on a couch
3619806638_7480883039.jpg: a man doing a trick on a skateboard
2550109269_bc4262bd27.jpg: a dog sitting on a bed with a pink blanket


In [5]:
human_references = {}

for img_name in validation_images:
    captions_list = captions_df[captions_df['image'] == img_name]['caption'].tolist()
    human_references[img_name] = captions_list

print("Human references loaded for", len(human_references), "images")
print("\nExample:")
example_img = validation_images[0]
print(f"Image: {example_img}")
for i, cap in enumerate(human_references[example_img], 1):
    print(f"  {i}. {cap}")

Human references loaded for 200 images

Example:
Image: 2594336381_a93772823b.jpg
  1. A little boy with a dirty face lays on a colorful rug .
  2. A little boy with a messy face sitting on a colorful blanket .
  3. little boy in red shirt and gray shorts with messy face
  4. The little boy is laying back relaxing on the colorful blanket .
  5. There is a little boy in a red shirt and a dirty face and he is smiling .


In [6]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method4

bleu_scores = []

for img_name in validation_images:
    prediction = blip_predictions[img_name].split()
    references = [ref.split() for ref in human_references[img_name]]
    score = sentence_bleu(references, prediction, smoothing_function=smoothie)
    bleu_scores.append(score)

avg_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"Average BLEU score over {len(bleu_scores)} images: {avg_bleu:.4f}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Average BLEU score over 200 images: 0.1919


In [7]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores, rouge2_scores, rougeL_scores = [], [], []

for img_name in validation_images:
    prediction = blip_predictions[img_name]
    # Use the best-matching reference (highest score) among the 5 human captions
    best_scores = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
    for ref in human_references[img_name]:
        scores = scorer.score(ref, prediction)
        for key in best_scores:
            if scores[key].fmeasure > best_scores[key]:
                best_scores[key] = scores[key].fmeasure
    rouge1_scores.append(best_scores['rouge1'])
    rouge2_scores.append(best_scores['rouge2'])
    rougeL_scores.append(best_scores['rougeL'])

avg_rouge1 = sum(rouge1_scores) / len(rouge1_scores)
avg_rouge2 = sum(rouge2_scores) / len(rouge2_scores)
avg_rougeL = sum(rougeL_scores) / len(rougeL_scores)

print(f"Average ROUGE-1: {avg_rouge1:.4f}")
print(f"Average ROUGE-2: {avg_rouge2:.4f}")
print(f"Average ROUGE-L: {avg_rougeL:.4f}")

Average ROUGE-1: 0.5556
Average ROUGE-2: 0.3198
Average ROUGE-L: 0.5337


In [8]:
import pandas as pd

results_summary = pd.DataFrame({
    "Metric": ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Score": [avg_bleu, avg_rouge1, avg_rouge2, avg_rougeL]
})

print("Baseline Evaluation Report: Zero-Shot BLIP on Flickr8k (200 validation images)\n")
print(results_summary.to_string(index=False))

# Save to CSV for the report / GitHub
results_summary.to_csv("baseline_evaluation_metrics.csv", index=False)
print("\nSaved as baseline_evaluation_metrics.csv")

Baseline Evaluation Report: Zero-Shot BLIP on Flickr8k (200 validation images)

 Metric    Score
   BLEU 0.191947
ROUGE-1 0.555609
ROUGE-2 0.319847
ROUGE-L 0.533728

Saved as baseline_evaluation_metrics.csv


## Baseline Evaluation Report: Zero-Shot BLIP on Flickr8k

**Validation subset:** 200 randomly sampled images (seed=42) from Flickr8k dataset.

| Metric   | Score  |
|----------|--------|
| BLEU     | 0.1919 |
| ROUGE-1  | 0.5556 |
| ROUGE-2  | 0.3198 |
| ROUGE-L  | 0.5337 |

### Interpretation:

- **BLEU (0.19)**: This is a relatively low score, which is expected for zero-shot
  captioning. BLEU is strict about exact word-order and n-gram overlap, and BLIP's
  captions tend to be short and generic compared to the more detailed human captions,
  which lowers BLEU significantly.
  
- **ROUGE-1 (0.56)**: This measures word-level (unigram) overlap and is much higher
  than BLEU. It shows that BLIP does capture many of the *right individual words*
  (e.g., "dog," "running," "man") even if the overall sentence structure differs
  from human captions.

- **ROUGE-2 (0.32)**: Bigram (2-word phrase) overlap is lower than ROUGE-1, which
  is expected — BLIP occasionally matches individual words correctly but does not
  always produce them in the same short phrases as human annotators.

- **ROUGE-L (0.53)**: This considers the longest common subsequence, so it reflects
  how well BLIP preserves the general order/structure of ideas — a fairly strong
  score, suggesting decent structural alignment with human captions.

### Overall Conclusion:
Zero-shot BLIP performs reasonably well at identifying key objects and actions
(reflected in high ROUGE-1), but struggles with generating longer, detailed, and
precisely-ordered phrases that match human captioning style (reflected in lower
BLEU and ROUGE-2). This establishes a **baseline** for comparison — future fine-tuning
(Day 3 onward) is expected to improve these scores, especially BLEU and ROUGE-2,
by adapting the model's vocabulary and phrasing style to the Flickr8k dataset.